<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">

# MONAI Label Radiology App - OHIF
In this notebook, we show the end-to-end setup of the web-based OHIF viewer and MONAI Label plugin. 

<p >
    <img src="./images/monailabel_arch_ohif.png" width="1000"/>
</p>

---
> **Note**: This notebook is an adaptation of the [monailabel_radiology_spleen_segmentation_OHIF.ipynb](https://github.com/Project-MONAI/tutorials/blob/main/monailabel/monailabel_radiology_spleen_segmentation_OHIF.ipynb) 
notebook released with MONAI Toolkit. 
We have removed cells for setup and data download which we have already loaded for you in this lab. 
---

<img src="./png4ohif/ohifV3_0.png" width="900"/>



### Table of Contents
**[0.0 OHIF Setup](#0.0-OHIF-Setup)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[0.1 OHIF Prerequisites](#0.1-OHIF-Prerequisites) <br>
**[1.0 Orthanc DICOM Server](#1.0-Orthanc-DICOM-Server)** <br>
**[2.0 Load DICOM Data Into Orthanc](#2.0-Load-DICOM-Data-Into-Orthanc)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.1 Upload Local Data](#2.1-Upload-Local-Data) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.2 Upload Data Through the UI](#2.2-Upload-Data-Through-the-UI) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.3 Download Data from TCIA Into Orthac](#2.3-Download-Data-from-TCIA-Into-Orthac) <br>
**[3.0 MONAI Label with OHIF Viewer](#3.0-MONAI-Label-with-OHIF-Viewer)** <br>
**[4.0 Start MONAI Label Server with Multi-Organ Segmentation](#4.0-Start-MONAI-Label-Server-with-Multi-Organ-Segmentation)** <br>
**[5.0 Auto-Segmentation with OHIF](#5.0-Auto-Segmentation-with-OHIF)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[5.1 MONAI Label Plugin](#5.1-MONAI-Label-Plugin) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[5.2 Run Auto-Segmentation](#5.2-Run-Auto-Segmentation) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[5.3 Edit Annotations and Submit Label](#5.3-Edit-Annotations-and-Submit-Label) <br>
**[6.0 Training and Active Learning](#6.0-Training-and-Active-Learning)** <br>
**[7.0 OHIF Conclusion](#7.0-OHIF-Conclusion)** <br>

---
# 0.0 OHIF Setup
Let's import some libs used in this notebook.

In [ ]:
import ipywidgets
import threading
import os
HOST_NAME=os.getenv('HOST_NAME')
HOST_IP=os.getenv('HOST_IP')
DLI_DATA_DIR=os.getenv('DLI_DATA_DIR')
os.chdir(DLI_DATA_DIR)

if len(HOST_NAME)==0:
    from requests import get
    HOST_NAME=get('https://api.ipify.org').text
    HOST_IP=HOST_NAME
print(f" HOST_NAME is {HOST_NAME} with IP {HOST_IP}")

## 0.1 OHIF Prerequisites
This DLI lab has MONAI Label and Orthanc already installed, along with models and dataset. 

---
# 1.0 Orthanc DICOM Server
[Orthanc](https://www.orthanc-server.com/) is a Belgian, open-source, lightweight [DICOM](https://www.dicomstandard.org/) server for healthcare and medical research.
You should read more about Orthanc at their site. 

In a nutshell, 
Orthanc aims at providing a simple, yet powerful, standalone DICOM server. It is designed to improve the DICOM flows in hospitals and to support research about the automated analysis of medical images. Orthanc lets its users focus on the content of the DICOM files, hiding the complexity of the DICOM format and of the DICOM protocol.

Orthanc is already set up for you and running as its own docker container.
Run the cell below to go to Orthanc server webpage.


In [ ]:
url_text='<h1><center><a href="http://'+HOST_NAME+'/pacs/" target="_blank">'
url_text=url_text+'<img src="https://upload.wikimedia.org/wikipedia/commons/c/c9/OrthancLogo.png" width="200" height="100"/>'
url_text=url_text+'<br>Click to open Orthanc <br> in a new window</a></center></h1>'
ipywidgets.HTML(url_text)

<div class="alert alert-block alert-info">
<b>Username:</b>orthanc<br>
<b>Password:</b>orthanc
</div>

The index page of Orthanc DICOM Web page:
<p align = "center"><img src="./png4ohif/ohif1.png" alt="drawing" width="900"/></p>


---
# 2.0 Load DICOM Data Into Orthanc
We need to have some DICOM data loaded into Orthanc.
We will use a public dataset. In the following section we will convert local NIFTI data to DICOM, then upload it.
You can also download data directly from TCIA or use your own data and upload it from the UI. 

## 2.1 Upload Local Data 

### 2.1.1 Convert NIFTI to DICOM Format
We have downloaded the spleen CT dataset in this lab. 
To use the DICOM server, the NIFTI formatted data needs to be converted to DICOM format. [Plastimatch](https://plastimatch.org/) is an open tool to generate simple DICOM data from a NIFTI file. The following commands show how to convert a sample.

```python
##### convert one of the NIFTI images to DICOM: name: <patient1>, output folder: <dicom_output>
!mkdir -p DICOMs
for i in ["1","11","15","23","30"]:
    print(f"converting spleen {i}")
    inputPath="datasets/Task09_Spleen/imagesTs/spleen_"+i+".nii.gz"
    outputPath="DICOMs/spleen_"+i
    patientID="patient"+i
    !plastimatch convert --study-description $patientID --patient-name $patientID --patient-id $patientID --input $inputPath --output-dicom $outputPath


### 2.1.2 Push DICOM Through `storescu`

In [ ]:
! ls DICOMs/

In [ ]:
# !python -m pynetdicom storescu dli-orthanc 4242 DICOMs -aet MONAILABEL -r
!python -m pynetdicom storescu dli-orthanc 4242 DICOMs/spleen_1* -aet MONAILABEL -r

## 2.2 Upload Data Through the UI
You can upload your DICOM data through the Orthanc user interface.
- Click `Upload` on the top right corner of Orthanc index page. 

<p align = "center"><img src="./png4ohif/ohif2.png" alt="drawing" width="900"/></p>

- Click `Select files to upload` and select **all** DICOM files of each subject.
- Click `Start the upload` to upload to the Orthanc server.
- Users can check the uploaded DICOM file for each subject by clicking `All studies` on the index page.

<p align = "center"><img src="./png4ohif/ohif3.png" alt="drawing" width="900"/></p>

## 2.3 Download Data from TCIA Into Orthac
Orthanc has a TCIA plugin which allows you to download data directly from TCIA. 
<p align = "center"><img src="./images/orthanc_TCIA1.png" width="600"/></p>

You can either go to TCIA site to seardch for images and download the NBIA cart as cvs or you can simply search the TCIA collection.
Here we will simply search for LIHC dataset which containes liver patients. 

<p align = "center"><img src="./images/orthanc_TCIA_collection.png" width="600"/></p>

We would need a patient with multiple scans so you can filter on patient: as below 
select CT images and click import

<p align = "center"><img src="./images/orthanc_TCIA_import.png" width="600"/></p>

---
# 3.0 MONAI Label with OHIF Viewer
[OHIF](https://ohif.org/) is Open Health Image Foundation. 
It is an extensible web imaging platform for medical devices.
You can create custom workflows with user-friendly interfaces.
Review cases and report results quickly, with zero installation required.

In [ ]:
url_text='<h1><center><a href="http://'+HOST_NAME+':8000/ohif/" target="_blank">'
url_text=url_text+'<img src="https://ohif.org/static/c99ccbad57599dbf9f3490519c9b444f/2f9b6/ohif-logo-dark.webp" width="200" height="100"/>'
url_text=url_text+'<br>Do NOT Click Now.<br> Only click after you start MONAI Label server (cell below) </a></center></h1>'
# print (url_text)
ipywidgets.HTML(url_text)

---
# 4.0 Start MONAI Label Server with Multi-Organ Segmentation

The command below will start MONAI Label server with all available models. If the user wants to start with a specific model, set the model name in the **--conf models** argument.

In [ ]:
#export MONAI_LABEL_DICOMWEB_USERNAME=orthanc
%env MONAI_LABEL_DICOMWEB_USERNAME=orthanc
#export MONAI_LABEL_DICOMWEB_PASSWORD=orthanc
%env MONAI_LABEL_DICOMWEB_PASSWORD=orthanc
## to disable DICOM to NIFTI conversion for faster performance
#!export MONAI_LABEL_DICOMWEB_CONVERT_TO_NIFTI=false
%env MONAI_LABEL_DICOMWEB_CONVERT_TO_NIFTI=false

os.chdir(DLI_DATA_DIR)
# !monailabel start_server --app apps/radiology --studies http://dli-orthanc:8042/dicom-web --conf models segmentation_spleen
!monailabel start_server \
    --app apps/radiology \
    --studies http://dli-orthanc:8042/dicom-web \
    --conf models segmentation

After the monailabel server is up, you can click the link above to go to the OHIF server page.

Note: By default, the MONAI Label server is up with port 8000.  If port 8000 is occupied by another application, users can specify the port number using `--port <port #>` in `monailabel start_server` command.

- Open the OHIF viewer.
- Select a patient. 
- Click on MONAI Label to open the 3d viewer.


<p align = "center"><img src="./png4ohif/ohifV3_4.png" alt="drawing" width="900"/></p>


---
# 5.0 Auto-Segmentation with OHIF

Below is a step-by-step tutorial on using OHIF for spleen auto-segmentation. A pretrained model is provided by MONAI Label.


## 5.1 MONAI Label Plugin

- MONAI Label plugin is already build-in. Click the subject to annotate.  On the labeling page, click the `MONAI Label` button in the top-right corner.
You may also need to click the refresh button next to the server address.

<p align = "center"><img src="./png4ohif/ohifV3_5.png" alt="drawing" width="900"/></p>


## 5.2 Run Auto-Segmentation

- MONAI Label has loaded the pretrained weights. Click the **Run** button in the right MONAI Label plugin panel. 

- Note: the pretrained models will be downloaded when starting the MONAI Label server in the "**model**" folder. For instance, in this use case, the downloaded pretrained model is saved at "**apps/radiology/model/pretrained_segmentation_spleen.pt**".

Users can monitor the logs in the MONAI Label server terminal.

The auto-segmentation results can show up as below:

<p align = "center"><img src="./png4ohif/ohifV3_6.png" alt="drawing" width="300"/></p>


## 5.3 Edit Annotations and Submit Label

- Users can edit the inference labels manually using the OHIF segmentation editor, the tool for adjusting annotation. 
Once done annotating, you should export DICOM SEG from the Segmentation click `...`,  then click  **Export DICOM SEG** on the right plugin panel to save the annotation.


<p align = "center"><img src="./png4ohif/ohifV3_7.png" alt="drawing" width="300"/></p>

---
# 6.0 Training and Active Learning

Active learning and interactive fine-tuning models have highlighted features in MONAI Label. Users can train their models any time when new annotated labels are saved. Click the **Update Model** button; MONAI Label server will fetch the saved final ground truth label and fine-tune the prior model. 

Advanced Feature: Users can set training parameters in the `options` panel.

<p align = "center"><img src="./png4ohif/ohif8.png" alt="drawing" width="400"/></p>

Users can do auto-segmentation using the latest fine-tuned model. 

<p align = "center"><img src="./png4ohif/ohif9.png" alt="drawing" width="400"/></p>

When next image is selected, the already trained images will be marked as labeled data, then won't be selected for the next labeling batch. Active learning strategies such as "first/random" will be used for selecting which unlabeled data to fetch. 

---
# 7.0 OHIF Conclusion

In this lab you learned:

- Basic usage of MONAI Label and OHIF
- How to set up the DICOM web server: Orthanc
- How to set up OHIF and how to start `monailabel` server
- Basic use cases of auto-segmentation, model training and active learning

<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">